In [26]:
import openai
from openai import OpenAI
import os
import json
import tempfile
import shutil
import re
from tqdm import tqdm
import pandas as pd
import requests
import time

In [2]:

# Set your OpenAI API key
openai.api_key = ""  # Replace with your actual key


client = OpenAI(api_key = "")

# Paths to the files
dataset_file_id = "file-SWQF3xk9nLcu9E7RKSn6Uq"

In [9]:
def extract_drive_file_id(drive_url):
    match = re.search(r"id=([a-zA-Z0-9_-]+)", drive_url)
    return match.group(1) if match else None

In [14]:
def getSubmissionsGDRIVE(
    excel_path,
    save_folder,
    student_id_col="Student ID",
    link_col="Upload your file here"
):
    """
    Reads Google Drive links from Excel, downloads PDFs,
    and returns {student_id: file_path}
    """

    os.makedirs(save_folder, exist_ok=True)
    df = pd.read_excel(excel_path)

    file_dict = {}

    for _, row in tqdm(df.iterrows(), 'Loading Submissions'):
        student_id = str(row[student_id_col]).replace("/", "_")
        drive_link = row[link_col]

        if pd.isna(drive_link):
            continue

        file_id = extract_drive_file_id(drive_link)
        if not file_id:
            continue

        download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
        response = requests.get(download_url, stream=True)

        if response.status_code != 200:
            continue

        filename = f"{student_id}.pdf"
        file_path = os.path.join(save_folder, filename)

        with open(file_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=32768):
                f.write(chunk)

        file_dict[student_id] = file_path

    return file_dict

In [22]:
adventureworks_rubric_prompt = '''You are an expert business analyst and instructor. Your task is to grade a student's multi-perspective business analysis assignment.

The student has submitted a **PDF file** containing:
1. A **Power BI report** (max 5 pages) with dashboards and visual analysis.
2. A **summary report** (max 2 pages) describing perspective, assumptions, methodology, and insights.

---

### Dataset Context

- **Dataset:** AdventureWorks 2022 (Excel version from CTU/Kaggle)
- **Business:** Bicycle manufacturing and sales company
- **Areas included:** 
  - Sales: orders, customers, territories, stores, salespeople, special offers
  - Customers / CRM: customer contacts, emails, phone numbers, shopping cart items
  - Operations / Production: products, categories, inventory, work orders, locations, bill of materials
  - HR / Employees: employee info, department, pay, shifts
  - Purchasing / Vendors: vendors, purchase orders, shipping methods, product supply

**Key schema relationships to check technical logic**:
- `SalesOrderHeader` → `SalesOrderDetail`
- `SalesOrderDetail` → `Product`
- `Product` → `ProductCategory`
- `Employee` → `Department` via `EmployeeDepartmentHistory`
- `PurchaseOrderHeader` → `PurchaseOrderDetail` → `ProductVendor` → `Vendor`
- Ensure date, currency, and quantity fields are correctly joined where applicable.

---

### Pre-Grading Instructions

1. Transcribe / summarize PDF content first:
   - Identify dashboard/page titles
   - Describe chart types (bar, line, table, KPI, etc.)
   - Extract axes, legends, filters, and KPIs where visible
   - Capture any visible metric names or calculations
   - Summarize any key findings mentioned in the student's PDF before grading.

   If visuals or text are unclear or unreadable, explicitly state this.

2. Edge Case Handling:
   - If page limits are exceeded, deduct proportionally
   - If joins or calculations are clearly broken or unsupported by evidence, deduct points
   - If sections are missing, assign zero for that criterion

---

### Grading Criteria (with justification)

1. **Relevance of Perspective (0-15 pts)**
   - Clear business focus (Sales, Customer Service, Operations, HR, Purchasing)
   - Logical cross-sectional analysis if multiple perspectives
   - **Output should include a justification field explaining why the score was assigned**

2. **Correctness of Analysis (0-25 pts)**
   - Proper relationships and joins
   - Accurate calculations and metrics
   - Schema-aware checks as noted above
   - Include justification for points awarded

3. **Visualization Clarity (0-20 pts)**
   - Dashboards are readable, intuitive, and effective
   - Charts, tables, KPIs communicate insights
   - Include justification

4. **Insights & Recommendations (0-20 pts)**
   - Actionable insights supported by evidence
   - Recommendations aligned with business objectives
   - Include justification

5. **Methodology & Assumptions (0-15 pts)**
   - Assumptions are clear
   - Methodology logical and reproducible
   - Include justification

6. **Presentation & Formatting (0-5 pts)**
   - Page limits followed
   - PDF properly merged
   - Include justification

---

### Output Format (JSON) - Output must only be json in the format below

```json
{
  "grades": {
    "relevance_of_perspective": {"score": 0-15, "justification": "text"},
    "correctness_of_analysis": {"score": 0-25, "justification": "text"},
    "visualization_clarity": {"score": 0-20, "justification": "text"},
    "insights_and_recommendations": {"score": 0-20, "justification": "text"},
    "methodology_and_assumptions": {"score": 0-15, "justification": "text"},
    "presentation_and_formatting": {"score": 0-5, "justification": "text"}
  },
  "total_score": 0-100,
  "feedback": {
    "strengths": "Brief summary of strongest points",
    "areas_for_improvement": "Brief summary of areas to improve"
  }
}

'''

In [19]:
def upload_files(files, assignment):
    file_ids = {}
    failed = {}
    for owner, file in tqdm(files.items(), 'Uploading Files...'):
        # Get the original file name and add the prefix
        original_name = os.path.basename(file)
        prefixed_name = assignment.replace('.json', '_') + original_name

        # Create a temporary file with the prefixed name
        temp_dir = tempfile.mkdtemp()
        temp_path = os.path.join(temp_dir, prefixed_name)
        shutil.copy(file, temp_path)  # Copy content to the new prefixed file

        # Upload the prefixed file
        with open(temp_path, "rb") as f:
            try:
                response = openai.files.create(file=f, purpose="assistants")
                file_id = response.id
            except Exception as e:
                failed[owner] = str(e)
            file_ids[owner] = file_id
        
        # print(f"📘 Uploaded '{prefixed_name}' for {owner}: {file_id}")

        # Clean up temp file
        shutil.rmtree(temp_dir)
    
    # Save the file IDs
    with open(assignment, "w") as f:
        json.dump(file_ids, f, indent=2)

    return file_ids, failed

NameError: name 'files' is not defined

In [29]:
def grade_submissions(student_file_ids, rubric_prompt, dataset_file_id=None):
    """
    Grades multiple student submissions using the OpenAI Responses API.
    
    Args:
        student_file_ids (dict): {student_id: file_id}
        rubric_prompt (str): JSON-output-enforcing rubric prompt
        dataset_file_id (str): Optional dataset file (if used)
    
    Returns:
        grading_results (dict): {student_id: parsed_json_result}
        failed (dict): {student_id: error_message}
        raw_responses (list): full OpenAI responses
    """
    
    grading_results = {}
    failed = {}
    graded = []
    raw_responses = []
    failed_count = 0

    for student_id, file_id in tqdm(student_file_ids.items(), desc='Grading assessments...'):
        if failed_count > 5:
            break
        try:
            # Build input list
            input_blocks = [
                {
                    "role": "user",
                    "content": [
                        {"type": "input_file", "file_id": file_id}
                    ]
                }
            ]

            # If you want to pass the dataset PDF
            # if dataset_file_id:
            #     input_blocks[0]["content"].append(
            #         {"type": "input_file", "file_id": dataset_file_id}
            #     )

            # Add rubric prompt
            input_blocks[0]["content"].append(
                {"type": "input_text", "text": rubric_prompt}
            )

            # API call
            response = client.responses.create(
                model="gpt-5.2",
                # model="gpt-4.1",
                # model="gpt-4o",
                input=input_blocks,
                temperature=0.3
            )

            # print(response.usage)
            raw_responses.append(response)

            # Extract model output text
            raw_text = response.output[0].content[0].text
            text_output = raw_text[len("```json"):].rstrip("```").strip() if raw_text.startswith("```json") else raw_text.strip()

            # Attempt to parse JSON safely
            try:
                result_json = json.loads(text_output)
                grading_results[student_id] = result_json
                graded.append(student_id)
                failed_count = 0
                time.sleep(3)
            except json.JSONDecodeError:
                # If the model returned extra explanation (rare)
                failed[student_id] = "Invalid JSON received"
                failed_count += 1
                time.sleep(3)
                continue

        except Exception as e:
            failed[student_id] = f"Error grading {student_id}: {e}"
            failed_count += 1
            time.sleep(3)

    return grading_results, graded, failed, raw_responses


In [15]:
submissions = getSubmissionsGDRIVE('Baraka 2025 Project Exam (Responses).xlsx', 'AltSchool/Baraka2025/AdventureWorks_Project')

Loading Submissions: 67it [03:27,  3.09s/it]


In [21]:
student_file_ids, failed_uploads = upload_files(submissions, 'adventureworks_project.json')

Uploading Files...: 100%|███████████████████████| 66/66 [01:39<00:00,  1.50s/it]


In [30]:
grading_results, graded, failed, raw_responses = grade_submissions(student_file_ids, adventureworks_rubric_prompt)

Grading assessments...: 100%|███████████████████| 66/66 [36:38<00:00, 33.31s/it]


In [42]:
with open("AltSchool/Baraka2025/AdventureWorks_Project_Results.json", "w", encoding="utf-8") as f:
    json.dump(grading_results, f, indent=4)